# Single-frequency forecasting quick start

This small, deterministic example forecasts two components and their total while enforcing the accounting identity

$$\mathrm{component\_a} + \mathrm{component\_b} = \mathrm{total}.$$

Install the package with `pip install macroframe-forecast` before running the notebook.

In [ ]:
import numpy as np
import pandas as pd
from sktime.forecasting.naive import NaiveForecaster

from macroframe_forecast import MFF

Create reproducible annual data, then mark the final four rows as unknown. `MFF` treats every missing value as a value to forecast. We retain the final total as a known endpoint (an "island") so reconciliation adjusts the component forecasts to meet it.

In [ ]:
rng = np.random.default_rng(42)
years = pd.RangeIndex(1990, 2022, name="year")
trend = np.arange(len(years))

observed = pd.DataFrame(
    {
        "component_a": 100 + 1.5 * trend + rng.normal(0, 0.5, len(years)),
        "component_b": 60 + 0.8 * trend + rng.normal(0, 0.5, len(years)),
    },
    index=years,
)
observed["total"] = observed["component_a"] + observed["component_b"]

forecast_horizon = 4
data = observed.copy()
data.iloc[-forecast_horizon:] = np.nan
data.loc[data.index[-1], "total"] = observed.loc[observed.index[-1], "total"]
data.tail(6)

Use `?` as the time wildcard in the identity. Setting `parallelize=False` keeps this small example simple, while a naive drift forecaster makes it quick to run.

In [ ]:
model = MFF(
    data,
    forecaster=NaiveForecaster(strategy="drift"),
    equality_constraints=["component_a_? + component_b_? - total_?"],
    parallelize=False,
    n_forecast_error=2,
)
forecast = model.fit()
forecast.tail(6)

`model.df1` contains the first-stage forecasts. The returned dataframe contains the reconciled forecasts, which meet the known endpoint and satisfy the identity.

In [ ]:
comparison = pd.concat(
    {"first_stage": model.df1, "reconciled": forecast},
    axis=1,
)
display(comparison.tail(forecast_horizon))

identity_error = forecast["component_a"] + forecast["component_b"] - forecast["total"]
assert forecast.iloc[-forecast_horizon:].notna().all().all()
assert np.allclose(identity_error, 0)
identity_error.tail(forecast_horizon)

In [ ]:
ax = observed["total"].plot(label="original total", linestyle="--")
forecast["total"].plot(ax=ax, label="forecast total")
ax.axvline(data.index[-forecast_horizon], color="black", linestyle=":", label="forecast start")
ax.set_ylabel("value")
ax.legend();